# Spark Structured Streaming & Auto Loader
### Databricks Data Engineer Associate — ILT Teaching Notebook

---

## Exam Weightage

| Section | Weight | Questions |
|---------|--------|-----------|
| Databricks Lakehouse Platform | 24% | ~11 |
| ETL with Spark SQL and Python | 29% | ~13 ← HIGHEST |
| **Incremental Data / Structured Streaming** | **20%** | **~9 ← THIS SESSION** |
| Production Pipelines | 16% | ~7 |
| Data Governance | 9% | ~5 |

> **Strategy:** Need ~32/45 correct (70%). Streaming is worth ~9 questions — master it.

---
## Chapters Covered
1. Batch vs Streaming Data — Concepts & Examples
2. Real-Time Data Flow Architecture
3. Spark Streaming vs Structured Streaming
4. Structured Streaming — Live Code Demo
5. Auto Loader — Schema Evolution & File Notification
6. Write Stream — Triggers & Output Modes
7. Fault Tolerance — Checkpointing & WAL

---
## Chapter 1 — Batch vs Streaming Data

### What is Streaming Data?

```
BATCH Data:
  Arrives on a schedule — weekly, daily, hourly
  Delays are ACCEPTABLE
  Example: weekly sales report — 2 hours late is fine

STREAMING Data:
  Arrives continuously — every second
  Delays of even 5 minutes are NOT acceptable
  Example: live cricket score, cab ETA, Amazon recommendations
```

### Real-Time vs Near Real-Time

| Type | Delay | Example |
|------|-------|--------|
| **Real-Time** | Sub-second | Live cricket score, stock ticker |
| **Near Real-Time** | 30 sec – 5 min | Cab ETA, e-commerce clickstream |
| **Batch** | Hours / days | Weekly orders report |

### The CRITICAL Rule

> **Streaming vs Batch = HOW you PROCESS, not how data arrives.**
> 
> If Kafka sends data every second but you process it once a week → it is BATCH.

### Business Example — Amazon vs Flipkart

```
You browse for a laptop on Amazon → don't buy immediately
Amazon processes your clickstream within 1 minute
Sends you a discount offer for the same laptop
If they wait 5 minutes → you've already bought it on Flipkart
That 1-minute window = near real-time processing
```

---
## Chapter 2 — Real-Time Data Flow Architecture

```
Application (Amazon website, Uber app)
       |
       v  every user click/action = one event
Stream Receiver / Message Broker
  (Kafka | Azure Event Hub | AWS Kinesis | GCP Pub/Sub)
       |
       v  buffers events, does NOT process
Persistent Storage  ← safety net
  (ADLS Gen2 | S3 | GCS)
       |
       v  Databricks mounts and reads as stream
Databricks (spark.readStream)
       |
       v
Delta Table (Bronze → Silver → Gold)
```

### Stream Receivers by Cloud
| Cloud | Service |
|-------|---------|
| Open Source | Apache Kafka |
| Azure | Event Hub |
| AWS | Kinesis |
| GCP | Pub/Sub |

### Why use ADLS between Event Hub and Databricks?

**Reason 1:** Databricks processes data in-memory — a network glitch loses everything. ADLS provides durable persistence.

**Reason 2:** Event Hub has a retention period (Basic tier: 1 day, Standard: 7 days). After that, data is deleted FOREVER. ADLS preserves it.

---
## Chapter 3 — Spark Streaming vs Structured Streaming

### Micro-Batch Concept

```
A trigger fires every N seconds (e.g., 10 seconds)
All data in those 10 seconds = one micro-batch
Next 10 seconds = another micro-batch
Resources are shared, not monopolised
```

### The Stateful Processing Problem (old Spark Streaming)

```
Micro-batch 1: Customer CUST-00123 placed 100 orders
Micro-batch 2: Customer CUST-00123 placed 70 more orders

Q: How does Spark know the previous count was 100?
→ It needs to store STATE

Old Spark Streaming:  Developer manually tracked state with updateStateByKey() — error-prone
Structured Streaming: Checkpoint location automatically stores state — Spark handles it
```

### Comparison

| Feature | Spark Streaming (Old) | Structured Streaming (New) |
|---------|----------------------|---------------------------|
| API | Low-level RDD | High-level DataFrame / SQL |
| Schema | No | Full support |
| SQL queries | Not supported | Fully supported |
| State management | Manual | Automatic (checkpoint) |
| Late data | No built-in | Watermarking |
| Recommended? | No — deprecated | Yes — current standard |

> **Always use Structured Streaming.** Spark Streaming (RDD-based) is deprecated.

### Save Game Analogy — Checkpointing

```
Video game: save at checkpoint 2 → game crashes at level 3
→ restart from checkpoint 2, NOT from the beginning

Structured Streaming: micro-batch 3 fails
→ Spark resumes from last checkpoint (batch 2)
→ only reprocesses batch 3
```

---
## Chapter 4 — Structured Streaming: Live Code Demo

### ADLS Setup

In [ ]:
# ─── ADLS Setup ───────────────────────────────────────────────────────────────
# WARNING: Do NOT push this notebook to GitHub with the real key
storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"  # ← replace before running

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# All ADLS paths — defined once, used throughout this notebook
raw_path          = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw"
bronze_path       = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"
stream_checkpoint = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/_checkpoints"
autoloader_base   = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/_autoloader"

print("Setup complete!")
print(f"  raw_path        : {raw_path}")
print(f"  bronze_path     : {bronze_path}")
print(f"  checkpoints     : {stream_checkpoint}")
print(f"  autoloader meta : {autoloader_base}")

### Step 1 — The WRONG Approach (spark.read)

In [ ]:
# ─── WRONG: spark.read = BATCH read ───────────────────────────────────────────
# This reads the file as static data — NOT streaming

batch_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load(f"{raw_path}/orders.csv")

# Check if this is streaming
print(f"Is streaming? {batch_df.isStreaming}")
# Output: Is streaming? False
# This is BATCH — it reads everything at once and stops

### Step 2 — Switch to spark.readStream (but schema problem)

In [ ]:
# ─── ATTEMPT: spark.readStream with inferSchema ────────────────────────────────
# This will FAIL — Structured Streaming cannot infer schema

try:
    stream_df = spark.readStream \
        .format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"{raw_path}/")

    print(f"Is streaming? {stream_df.isStreaming}")

except Exception as e:
    print(f"ERROR: {e}")
    print()
    print("WHY IT FAILS:")
    print("Structured Streaming cannot infer schema from streaming data.")
    print("It processes data in micro-batches — it never scans ALL files upfront.")
    print("Solution: define the schema manually.")

### Step 3 — Correct: Define Schema First, Then Stream

In [ ]:
# ─── CORRECT Step 1: Read one file as batch to get the schema ─────────────────
# Trick: load a sample batch first to extract the schema
# Then reuse that schema in the streaming read

sample_df     = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{raw_path}/orders.csv")
orders_schema = sample_df.schema

print("Schema extracted from orders.csv:")
print(orders_schema)

# orders.csv columns:
# OrderID, CustomerID, OrderDate, ShippingDate,
# ExpectedDeliveryDate, ActualDeliveryDate,
# ShippingTierID, SupplierID, OrderChannel

In [ ]:
# ─── CORRECT Step 2: Use extracted schema in readStream ───────────────────────

orders_stream = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(orders_schema) \
    .load(f"{raw_path}/")

print(f"Is streaming? {orders_stream.isStreaming}")
# Output: Is streaming? True
print()
print("Stream is ready. Columns:")
print(orders_stream.columns)

### Step 4 — Run Aggregations on the Stream

> The stream does NOT run until you call `writeStream` or `display()`.  
> `display()` in Databricks triggers the stream and shows a live updating result.

In [ ]:
# ─── Count orders by OrderChannel (streaming aggregation) ─────────────────────
# display() triggers the stream — results update live as new files arrive

from pyspark.sql.functions import count, col

orders_by_channel = orders_stream.groupBy("OrderChannel").agg(
    count("*").alias("order_count")
)

# Uncomment display() to run the live stream in Databricks:
# display(orders_by_channel)

# To STOP the stream: click the Stop button or run the interrupt
# The cluster must stay RUNNING for streaming to work
print("Stream query defined. Run display(orders_by_channel) in Databricks to start.")

### Step 5 — Schema Evolution Limitation

```
Problem: A new CSV arrives with an extra column (e.g., 'promo_code')
→ Structured Streaming SILENTLY IGNORES the new column
→ No error is thrown — the data for that column is just missing

Fix: manually update orders_schema to include the new column → restart stream

Real world: your upstream team MUST communicate schema changes in advance

Solution for auto schema evolution → Auto Loader (next chapter)
```

### Key Rules — Structured Streaming

| Rule | Detail |
|------|--------|
| Use `spark.readStream` | NOT `spark.read` |
| Schema MUST be defined manually | `inferSchema` does NOT work |
| `display()` triggers the stream | Stream won't run without an action |
| Cluster must stay RUNNING | Stream stops when cluster stops |
| Schema evolution NOT supported | New columns are silently ignored |
| Exactly-once processing | Each file/record processed only once |

---
## Chapter 5 — Auto Loader (`cloudFiles`)

### Why Auto Loader was created

```
Structured Streaming Problem:
  - No schema inference
  - No schema evolution (new columns silently ignored)
  - Must manually update schema for every change

Auto Loader Solution:
  - Auto-infers schema (stores it in schemaLocation)
  - Handles new columns automatically (schema evolution)
  - Databricks-proprietary feature (Premium workspace required)
```

### Required Parameters

| Parameter | What it does |
|-----------|-------------|
| `format("cloudFiles")` | Activates Auto Loader |
| `cloudFiles.format` | File type: `csv`, `json`, `parquet` |
| `cloudFiles.schemaLocation` | Where to store inferred schema — SEPARATE folder from data |
| `checkpointLocation` | Tracks which files were already processed |

### Schema Evolution Modes

| Mode | What happens when new column arrives |
|------|-------------------------------------|
| `addNewColumns` (default) | Auto Loader adds the new column to schema automatically |
| `rescue` | New column data stored in `_rescued_data` JSON column |
| `failOnNewColumns` | Pipeline FAILS — strict schema enforcement |

In [ ]:
# ─── Auto Loader: raw/orders.csv → Bronze (with schema inference) ──────────────
# Production-grade incremental ingestion
# All paths come from setup cell s-06

orders_autoloader_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format",          "csv") \
    .option("cloudFiles.schemaLocation",  f"{autoloader_base}/schemas/orders") \
    .option("header",                     "true") \
    .option("inferSchema",                "true") \
    .load(f"{raw_path}/")

# No schema needed — Auto Loader infers it and stores at:
#   abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/_autoloader/schemas/orders

print(f"Is streaming? {orders_autoloader_stream.isStreaming}")
print("Schema inferred automatically — no manual definition needed!")
print()
print("Columns found:")
print(orders_autoloader_stream.columns)

In [ ]:
# ─── Auto Loader with schema evolution mode ───────────────────────────────────
# rescue mode: if a new column appears, its data goes into _rescued_data JSON column

orders_rescue_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format",              "csv") \
    .option("cloudFiles.schemaLocation",      f"{autoloader_base}/schemas/orders_rescue") \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .option("header",                         "true") \
    .option("inferSchema",                    "true") \
    .load(f"{raw_path}/")

# When a new column (e.g., PromoCode) appears in a new orders file:
# _rescued_data will contain:
#   {"PromoCode": "SAVE10", "_rescued_data_file_path": ".../raw/orders_new.csv"}

print("Auto Loader with rescue mode — new columns go into _rescued_data")

# To extract rescued data:
# from pyspark.sql.functions import get_json_object
# orders_rescue_stream.withColumn("PromoCode", get_json_object(col("_rescued_data"), "$.PromoCode"))

In [ ]:
# ─── Write Auto Loader stream to Bronze Delta ─────────────────────────────────
# Checkpoint at: abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/_checkpoints/orders_autoloader
# Output at    : abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/bronze/orders_stream

from pyspark.sql.functions import current_timestamp, input_file_name

orders_with_meta = orders_autoloader_stream \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", input_file_name())

query = orders_with_meta.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{stream_checkpoint}/orders_autoloader") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start(f"{bronze_path}/orders_stream")

query.awaitTermination()

count = spark.read.format("delta").load(f"{bronze_path}/orders_stream").count()
print(f"Orders in Bronze via Auto Loader: {count:,} rows")

### File Notification vs Directory Listing

```
Directory Listing (default):
  Auto Loader scans ALL files in folder → finds which are new → processes them
  Works fine for millions of files
  Slower for billions of files (full scan each time)

File Notification:
  Cloud sends an event (Azure Event Grid / AWS SNS) when NEW file arrives
  Auto Loader gets notified directly → processes only that file
  No full scan needed — scales to billions of files
  Requires extra cloud setup (Event Grid subscription)
```

### Auto Loader vs Structured Streaming vs COPY INTO

| Feature | COPY INTO | Structured Streaming | Auto Loader |
|---------|-----------|---------------------|-------------|
| Scale | Small–Medium | Medium–Large | Large–Billions |
| Schema inference | Yes | No (manual) | Yes (automatic) |
| Schema evolution | No | No | Yes |
| File discovery | Directory listing | Directory listing | Directory + File notification |
| Workspace tier | Standard | Standard | Premium only |
| Cost | Lower | Medium | Higher |

---
## Chapter 6 — Write Stream: Triggers & Output Modes

### Write Stream Syntax

```python
streaming_df
  .writeStream
  .format('delta')
  .outputMode('append')
  .trigger(processingTime='10 seconds')
  .option('checkpointLocation', '/path/checkpoint/')
  .start('/path/output/')
```

### Why `.writeStream` not `.write`?

```
spark.read    → use .write
spark.readStream → MUST use .writeStream
```

### Trigger Types

| Trigger | Syntax | Behavior | Best For |
|---------|--------|----------|----------|
| Processing Time | `.trigger(processingTime='10 seconds')` | Fires every 10 sec | Continuous streaming |
| Available Now | `.trigger(availableNow=True)` | Process all pending, then stop | Incremental batch |
| Continuous | `.trigger(continuous='1 second')` | No batching, true stream | Ultra-low latency |
| Default (none) | No trigger | 500ms micro-batch | Not recommended (costly) |

> **Databricks recommendation:** Always specify an explicit trigger to control cost.  
> Default 500ms keeps the cluster constantly processing — expensive.

### Output Modes

| Mode | Behavior | Use Case |
|------|----------|----------|
| `append` | Insert new rows only, never updates | Bronze/Silver ingestion |
| `complete` | Overwrite entire output with fresh aggregated result | Monthly dashboard overwrite |
| `update` | Write only rows that CHANGED since last micro-batch | Efficient aggregated updates |

In [ ]:
# ─── Trigger examples (do not run all — pick one) ─────────────────────────────

# Example 1: Trigger every 10 seconds (continuous streaming)
q1 = orders_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .option("checkpointLocation", f"{stream_checkpoint}/orders_10sec") \
    .start(f"{bronze_path}/orders_trigger_demo")

# q1.stop()  # ← Stop when done

# Example 2: Process all available files and stop (incremental batch)
q2 = orders_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", f"{stream_checkpoint}/orders_available") \
    .start(f"{bronze_path}/orders_available_demo")

# q2.awaitTermination()  # Wait for this one-shot job to finish

# Write to a managed Delta TABLE (instead of path):
# .toTable('amazon_project.bronze.orders_stream')

print("Trigger examples defined — uncomment the one you want to run.")

---
## Chapter 7 — Fault Tolerance: Checkpointing & WAL

### Checkpoint Folder Structure

```
_checkpoints/orders_autoloader/
  metadata     ← stream query ID and configuration
  sources/     ← which files/offsets were read per micro-batch
  offsets/     ← current stream position (like a bookmark)
  commits/     ← log of completed micro-batch transactions
  state/       ← ONLY when GROUP BY / window functions are used
```

> The `state/` folder only appears when you use aggregation queries.

### Checkpoint = Memory for Spark Streaming

```
Micro-batch 1: CustomerID CUST-00123 → 100 orders  → checkpoint saved
Micro-batch 2: CustomerID CUST-00123 → 70 more     → checkpoint saved
Total count: 170

Micro-batch 3 FAILS:
  Spark reads checkpoint → resumes from count 170
  Only reprocesses batch 3 → no data loss
```

### WAL (Write-Ahead Log) = Idempotency

```
WAL = a log entry created for EACH record BEFORE it is processed

Before processing a record:
  Spark checks: "Does a WAL entry exist for this record?"
  YES → skip (already processed)
  NO  → process and create WAL entry

Result: Idempotency — running the pipeline 100 times = same output, no duplicates
```

### WAL = Attendance Register Analogy

```
Teacher checks register before student enters:
  Name already marked → student entered already, skip
  Name not marked    → mark it and let them in

WAL does the same:
  Already logged → skip (already processed)
  Not logged     → process and log it
```

### WAL vs Checkpoint — Key Distinction

| | Checkpoint | WAL |
|-|-----------|-----|
| Stores | STATE (aggregated counts) | LOG per record |
| Enables | Recovery + stateful processing | Idempotency (no duplicates) |
| Together | **Exactly-Once Processing guarantee** | |

### Certification Summary

```
Exam expects you to know:
  1. Auto Loader vs COPY INTO (scale + notification difference)
  2. Output modes: complete = overwrite, append = insert only
  3. Fault tolerance = Checkpointing + WAL
  4. Triggers: always specify explicit trigger; default = 500ms
  5. Medallion: raw→Bronze (Auto Loader/COPY INTO), Bronze→Silver (MERGE), Silver→Gold (GROUP BY)
  6. stream.isStreaming = True only with readStream
  7. Schema must be manually defined in Structured Streaming (not with Auto Loader)
```

In [ ]:
# ─── Verify checkpoint folder contents after running writeStream ───────────────
# Run this AFTER running a writeStream to see what Databricks creates
# Checkpoint location: abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/_checkpoints/orders_autoloader

checkpoint_path_to_check = f"{stream_checkpoint}/orders_autoloader"

try:
    files = dbutils.fs.ls(checkpoint_path_to_check)
    print("Checkpoint folder contents:")
    for f in files:
        print(f"  {f.name}  ← {f.size} bytes")
    print()
    print("Look for: metadata, sources/, offsets/, commits/")
    print("state/ only appears if you used GROUP BY / window functions")
except Exception as e:
    print(f"Run a writeStream first, then check the checkpoint folder.")
    print(f"Error: {e}")